In [37]:
import customfunctions as ctf
import pandas as pd
import numpy as np
import heartpy as hp
import neurokit2 as nk
import prepro as prep  # Custom functions for preprocessing
import os
import glob
from pathlib import Path
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Parameters

### Preprocessing parameters

In [38]:
# BVP preprocessing
bvp_prepro_winsz = 10
bvp_prepro_ovlap = 0.02
bvp_fs = 64

# EDA preprocessing
eda_fs = 4
iter_num = 7

### Processing parameters

In [39]:
basalscene = 0
targetscene = 3
# Note, subject 10 does not have scene 2 nor 3 data
subjects = [1, 2, 3, 4, 6, 8, 9]
winsz = 90
ovlap = 0.5
# Outlier removal method, either iqr_outlier, mod_zscore_outlier (both applicable in non-normal data) or winsorization (winsor_outlier)
outliermeth = 'iqr_outlier'
# Outlier treatment strategy, either elimination ('elim') or imputation ('impute')
outliertreatment = 'impute'
p_low=0.5
p_high=0.95
# Normalization type, either z-score ('zscore'), mean ('mean'), minmax scaling ('minmax'), robust scaling ('robust')
normtype = 'minmax'
base_dir = Path().resolve()

# Data import function

In [40]:
folderpath = os.path.join(base_dir, "..", "RawData")
folderpath = os.path.abspath(folderpath)


def dataimport(subjects, folderpath, scene):
    bvp_rawdata_list = []
    eda_rawdata_list = []

    for subject in subjects:
        subject_str = f"S{subject}" if subject == 10 else f"S{subject:02d}"
        pattern = os.path.join(
            folderpath,
            f"S{subject}",
            "Empatica",
            f"P300_{subject_str}R0{scene}*",
            "Raw",
        )
        matched_folders = glob.glob(pattern)
        raw_folder = matched_folders[0]
        bvp_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileBVP.csv")).drop(
            "Datetime", axis=1
        )
        eda_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileEDA.csv")).drop(
            "Datetime", axis=1
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_rawdata_list.append(bvp_dataholder)
        eda_rawdata_list.append(eda_dataholder)

    bvp_rawdata_df = pd.concat(bvp_rawdata_list, ignore_index=True)
    eda_rawdata_df = pd.concat(eda_rawdata_list, ignore_index=True)

    return bvp_rawdata_df, eda_rawdata_df


basal_bvp_rawdata, basal_eda_rawdata = dataimport(
    subjects, folderpath, scene=basalscene
)
tscene_bvp_rawdata, tscene_eda_rawdata = dataimport(
    subjects, folderpath, scene=targetscene
)

# Preprocessing function

In [41]:
def preprocess(subjects, bvp_df, eda_df, bvp_fs, eda_fs, winsz, ovlap, iter_num):
    bvp_prepdata_list = []
    eda_prepdata_list = []
    for subject in subjects:
        bvp_dataholder = pd.DataFrame(
            prep.preprocess_bvp(
                sig=bvp_df[bvp_df["Subject"] == subject]["valueBVP"],
                fs=bvp_fs,
                winsz=winsz,
                ovlap=ovlap,
            ),
            columns=["valueBVP"],
        )
        eda_dataholder = pd.DataFrame(
            prep.preprocess_eda(
                sig=eda_df[eda_df["Subject"] == subject]["valueEDA"],
                fs=eda_fs,
                iter_num=iter_num,
                verbose=False,
            )[0],
            columns=["valueEDA"],
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_prepdata_list.append(bvp_dataholder)
        eda_prepdata_list.append(eda_dataholder)

    bvp_prepdata_df = pd.concat(bvp_prepdata_list, ignore_index=True)
    eda_prepdata_df = pd.concat(eda_prepdata_list, ignore_index=True)

    return bvp_prepdata_df, eda_prepdata_df


basal_bvp_prepdata, basal_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=basal_bvp_rawdata,
    eda_df=basal_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)

tscene_bvp_prepdata, tscene_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=tscene_bvp_rawdata,
    eda_df=tscene_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)


# Processing function

In [42]:
import warnings
def get_bvp_feats(window, smooth_window, fs):
    wd = {}
    wd = hp.peakdetection.detect_peaks(
        window, smooth_window, ma_perc=20, sample_rate=fs
    )
    wd = hp.analysis.calc_rr(wd["peaklist"], sample_rate=fs, working_data=wd)
    wd = hp.peakdetection.check_peaks(
        wd["RR_list"], wd["peaklist"], window[wd["peaklist"]], working_data=wd
    )
    wd = hp.analysis.clean_rr_intervals(working_data=wd, method="quotient-filter")
    rr_list = wd["RR_list_cor"]
    rr_diff = np.diff(rr_list)
    rr_sqdiff = np.power(rr_diff, 2)
    wd, msrs = hp.analysis.calc_ts_measures(
        rr_list, rr_diff, rr_sqdiff, working_data=wd
    )
    wd, msrs = hp.analysis.calc_fd_measures(measures=msrs, working_data=wd)
    bvp_feats = pd.DataFrame([msrs])[
        [
            "bpm",
            "sdnn",
            'rmssd',
            "pnn50",
            "hr_mad",
            "lf",
            "hf",
            "lf/hf",
            "p_total",
            "lf_nu",
            "hf_nu",
        ]
    ]

    return bvp_feats


def get_eda_feats(window, fs):
    signals, info = nk.eda_process(window, fs)
    mean_eda = np.nanmean(window)
    mean_tonic = np.nanmean(signals["EDA_Tonic"])
    scr_peak_count = np.nansum(signals["SCR_Peaks"])
    scr_sum_amp = np.nansum(info["SCR_Amplitude"])
    scr_mean_amp = np.nanmean(info["SCR_Amplitude"])
    scr_mean_risetime = np.nanmean(info["SCR_RiseTime"])
    scr_mean_recoverytime = np.nanmean(info["SCR_RecoveryTime"])

    eda_feats = pd.DataFrame(
        [
            {
                "mean_eda": mean_eda,
                "mean_tonic": mean_tonic,
                "scr_peak_count": scr_peak_count,
                "scr_sum_amp": scr_sum_amp,
                "scr_mean_amp": scr_mean_amp,
                "scr_mean_risetime": scr_mean_risetime,
                "scr_mean_recoverytime": scr_mean_recoverytime,
            }
        ]
    )

    return eda_feats


def process(subjects, bvp_prepdata_df, eda_prepdata_df, bvp_fs, eda_fs, ovlap, winsz):
    feat_mat_list = []
    for subject in subjects:
        # BVP
        bvp_probe = np.array(
            bvp_prepdata_df[bvp_prepdata_df["Subject"] == subject]["valueBVP"]
        )
        bvp_smooth = uniform_filter1d(
            bvp_probe, size=int(0.75 * bvp_fs), mode="nearest"
        )
        bvp_windowed = ctf.timewindowpadded(
            data=bvp_probe, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        smooth_windowed = ctf.timewindowpadded(
            data=bvp_smooth, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        bvp_holder_list = []
        for windnum in range(0,bvp_windowed.shape[0]):
            bvp_feat_row = get_bvp_feats(
                window=bvp_windowed[windnum],
                smooth_window=smooth_windowed[windnum],
                fs=bvp_fs,
            )
            bvp_holder_list.append(bvp_feat_row)
        bvp_holder_df = pd.concat(bvp_holder_list, ignore_index=True)

        # EDA
        eda_probe = np.array(
            eda_prepdata_df[eda_prepdata_df["Subject"] == subject]["valueEDA"]
        )
        eda_windowed = ctf.timewindowpadded(
            data=eda_probe, fs=eda_fs, ovlap=ovlap, winsz=winsz
        )
        eda_holder_list = []
        for windnum in range(0,eda_windowed.shape[0]):
            eda_feat_row = get_eda_feats(window=eda_windowed[windnum], fs=eda_fs)
            eda_holder_list.append(eda_feat_row)
        eda_holder_df = pd.concat(eda_holder_list, ignore_index=True)

        valid_len = min(len(bvp_holder_df), len(eda_holder_df))
        joined_df = pd.concat([bvp_holder_df.iloc[:valid_len], eda_holder_df.iloc[:valid_len]], axis=1)
        joined_df.insert(loc=0, column="Subject", value=[subject] * len(joined_df))
        feat_mat_list.append(joined_df)
        
    feat_mat_df = pd.concat(feat_mat_list, ignore_index=True)
    return feat_mat_df


basal_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=basal_bvp_prepdata,
    eda_prepdata_df=basal_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

tscene_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=tscene_bvp_prepdata,
    eda_prepdata_df=tscene_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

warnings.filterwarnings('ignore')

In [43]:
basal_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,62.466476,60.745158,58.491992,0.136986,39.0625,1.020411e+03,1.069850e+03,0.953789,2.602937e+03,48.817396,51.182604,-0.005557,-0.005470,17,0.133718,0.007866,0.455882,0.750000
1,1,63.840399,74.530595,67.652525,0.126582,46.8750,9.577659e+02,8.816200e+02,1.086370,2.133140e+03,52.069872,47.930128,-0.000625,-0.000620,17,0.109199,0.006825,0.593750,0.950000
2,1,67.908512,101.847943,86.004657,0.121622,62.5000,2.391600e+03,2.279704e+03,1.049084,8.190353e+03,51.197694,48.802306,0.002401,0.002226,18,0.096585,0.005681,1.132353,1.100000
3,1,68.746803,86.251777,62.814230,0.145161,62.5000,8.017548e+02,1.627016e+03,0.492776,2.820222e+03,33.010728,66.989272,0.009608,0.009637,15,0.122990,0.008199,1.050000,1.027778
4,2,71.431793,83.631418,54.698887,0.102041,46.8750,1.013593e+03,8.258658e+02,1.227310,2.730259e+03,55.102794,44.897206,-0.023166,-0.023263,74,0.429396,0.005882,0.414384,0.535714
5,2,74.250000,78.236675,58.271328,0.091837,46.8750,5.973643e+02,8.025838e+02,0.744301,4.640375e+03,42.670461,57.329539,0.001783,0.001801,78,0.461093,0.005911,0.442308,0.632353
6,2,77.111562,85.465727,64.847496,0.102041,62.5000,6.725265e+02,3.094232e+02,2.173484,2.605237e+03,68.488895,31.511105,0.002389,0.001928,26,0.402167,0.015468,0.557692,0.972222
7,2,85.404504,106.789867,111.293637,0.189873,54.6875,2.977515e+03,1.315640e+03,2.263169,6.425113e+03,69.354939,30.645061,-0.000033,0.000183,22,0.376104,0.017096,0.579545,0.966667
8,3,81.923703,56.356810,50.081731,0.128205,31.2500,8.102969e+02,1.374918e+03,0.589342,2.323080e+03,37.080880,62.919120,0.000878,0.000861,35,0.156139,0.004461,0.414286,0.550000
9,3,85.128820,55.584121,51.950588,0.100000,31.2500,5.423301e+02,8.034705e+02,0.674984,1.635130e+03,40.297951,59.702049,-0.000713,-0.000727,36,0.211838,0.005884,0.416667,0.479167


In [44]:
tscene_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,164.365971,132.219350,149.818175,0.303797,85.9375,1296.090196,2059.325665,0.629376,3355.415861,38.626813,61.373187,0.009317,0.009089,20,0.303490,0.015175,0.650000,0.982143
1,1,183.067203,117.753425,163.549030,0.292683,78.1250,665.331751,1607.125991,0.413989,2272.457742,29.278069,70.721931,0.003045,0.003111,26,0.280132,0.010774,0.480769,0.750000
2,1,192.642140,96.986942,118.198472,0.258427,62.5000,114.327087,2356.942880,0.048507,2471.269967,4.626248,95.373752,0.004435,0.004640,16,0.303442,0.018965,0.515625,0.477273
3,1,152.102377,151.589901,139.412634,0.296875,109.3750,985.405568,4644.389513,0.212171,5629.795081,17.503400,82.496600,0.026684,0.026693,5,0.205959,0.041192,0.650000,5.500000
4,2,91.596330,117.800725,146.432282,0.218750,31.2500,757.549784,3482.011957,0.217561,4239.561741,17.868587,82.131413,-0.007472,-0.006858,9,0.306149,0.034017,1.222222,3.000000
5,2,92.530120,129.083685,182.116099,0.245283,39.0625,293.682833,2727.169537,0.107688,3020.852370,9.721853,90.278147,-0.007493,-0.007410,10,0.480404,0.048040,1.475000,2.305556
6,2,103.572519,160.061100,199.674841,0.250000,62.5000,550.521600,5334.521355,0.103200,5885.042955,9.354589,90.645411,-0.003431,-0.004394,27,0.342504,0.013173,0.538462,0.630952
7,2,100.702622,157.484444,177.532013,0.211538,46.8750,1723.003999,5999.462752,0.287193,7722.466751,22.311575,77.688425,-0.023351,-0.023370,20,0.437738,0.021887,0.587500,0.783333
8,3,109.990557,121.862416,115.772710,0.122222,62.5000,3340.782521,6571.043318,0.508410,9911.825839,33.705016,66.294984,-0.021928,-0.018657,6,1.344445,0.224074,1.458333,3.625000
9,3,110.678383,112.241643,115.461179,0.135802,39.0625,1646.535178,4537.992536,0.362833,6184.527714,26.623459,73.376541,-0.013049,-0.012558,15,0.420786,0.028052,0.950000,1.318182


In [45]:
def remove_outliers_iqr(col, treatment='elim'):
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    if treatment == 'impute':
        return col.clip(lower=lower, upper=upper)
    else:
        return col[(col >= lower) & (col <= upper)]

def remove_outliers_modz(col, treatment='elim'):
    x_mid = col.median()
    mad = abs(col - x_mid).median()
    mz = 0.6745 * (col - x_mid) / mad

    if treatment == 'impute':
        return col.mask(abs(mz) > 3.5, x_mid)
    else:
        return col[abs(mz) <= 3.5]

def remove_outliers_winsor(col, p_low=0.05, p_high=0.95):
    lower = col.quantile(p_low)
    upper = col.quantile(p_high)
    return col.clip(lower=lower, upper=upper)


def norm_with_ref(subjects, feat_mat, ref_mat, normtype=str, outliermeth=outliermeth, outliertreatment=outliertreatment):
        # This line is redundant, eliminate and substitute application on this cell based on preference.
    channel_cols = [
        col for col in ref_mat.columns if col not in ["Subject"]
    ]

    normdata_list = []

    # Data is normalized per subject   ####### Corregir
    for subject in subjects:
        data = feat_mat[feat_mat["Subject"] == subject].copy()
        
        if outliermeth == 'iqr_outlier':
            def outlier_func(col):
                return remove_outliers_iqr(col, outliertreatment)
        elif outliermeth == 'mod_zscore_outlier':
            def outlier_func(col):
                return remove_outliers_modz(col, outliertreatment)
        elif outliermeth == 'winsor_outlier':
            def outlier_func(col):
                return remove_outliers_winsor(col, p_low=0.5, p_high=0.95)

        data[channel_cols] = data[channel_cols].apply(outlier_func)

        ref_data = ref_mat[ref_mat["Subject"] == subject].drop("Subject", axis=1)
        ref_data = ref_data.apply(outlier_func).dropna()

        if normtype == 'zscore':
            ref_mean = np.nanmean(ref_data)
            ref_std = np.nanstd(ref_data)
            data[channel_cols] = (data[channel_cols] - ref_mean) / ref_std
        elif normtype == 'mean':
            ref_mean = np.nanmean(ref_data)
            data[channel_cols] = (data[channel_cols] - ref_mean) / ref_mean
        elif normtype == 'minmax':
            scaler = MinMaxScaler(feature_range=(-0.5,0.5))
            scaler.fit(ref_data[channel_cols])
            scaled = scaler.transform(data[channel_cols])
            #scaled = np.clip(scaled, 0, 1)
            data[channel_cols] = scaled
        elif normtype == 'robust':
            scaler = RobustScaler()
            scaler.fit(ref_data[channel_cols])
            data[channel_cols] = scaler.transform(data[channel_cols])
        normdata_list.append(data.dropna())
    normdata_df = pd.concat(normdata_list, ignore_index=True)

    return normdata_df

norm = norm_with_ref(subjects=subjects, feat_mat=tscene_feat_mat, ref_mat=basal_feat_mat, normtype=normtype)
norm

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,15.725190,1.238914,2.819423,7.239100,1.500000,-0.097486,0.342371,-0.291142,-0.231103,-0.289414,0.289414,0.480858,0.463736,1.261905,5.072079,3.270316,-0.213043,0.163265
1,1,18.702938,0.886968,3.318497,6.766938,1.166667,-0.611083,0.018929,-0.664017,-0.469350,-0.838399,0.838399,0.067204,0.068042,3.547619,4.443029,1.522701,-0.463211,-0.500000
2,1,20.227530,0.381735,1.670145,5.480003,0.500000,-1.059740,0.555246,-1.296734,-0.425612,-2.286023,2.286023,0.158897,0.169225,-0.261905,5.070764,4.775808,-0.411685,-1.279221
3,1,13.772490,1.710185,2.441214,6.945023,2.500000,-0.350462,2.018430,-1.013401,0.269254,-1.529840,1.529840,1.626065,1.629026,-4.452381,2.445486,13.206699,-0.213043,9.517973
4,2,0.943137,1.061875,1.320780,1.341298,-1.500000,-0.426637,2.652987,-0.846798,-0.072152,-1.429449,1.429449,0.002748,0.034472,-0.732143,-1.323109,2.008978,4.391193,5.145455
5,2,1.009967,1.507293,2.029054,1.726247,-1.000000,-0.639082,1.902809,-0.919137,-0.391197,-1.734748,1.734748,0.001659,0.005226,-0.714286,0.727216,3.259595,5.921678,3.554545
6,2,1.800250,2.730192,2.377571,1.794683,0.500000,-0.521453,4.494051,-0.922092,0.358616,-1.748511,1.748511,0.206504,0.165005,-0.410714,-0.895353,0.150205,0.251251,-0.281818
7,2,1.594857,2.628473,1.938067,1.236670,-0.500000,0.015529,5.154885,-0.800954,0.839632,-1.262949,1.262949,-0.576149,-0.616126,-0.535714,0.225198,0.927275,0.548162,0.067273
8,3,0.446438,0.045311,0.111674,0.141975,1.100000,1.546670,1.194975,-0.219871,1.068395,-0.173009,0.173009,-4.355830,-3.679704,-1.608527,12.073426,73.782848,7.459571,10.714084
9,3,0.469632,-0.033845,0.108773,0.534294,-0.100000,0.317137,0.597502,-0.822117,0.362089,-0.895896,0.895896,-2.692904,-2.558288,-1.166667,2.300215,7.479604,3.584158,2.490873


In [46]:
channel_cols = [col for col in basal_feat_mat.columns if col not in ["Subject"]]
basal_outlier_treated = basal_feat_mat
basal_outlier_treated[channel_cols] = basal_feat_mat[channel_cols].apply(remove_outliers_winsor, args=(0.1,0.9,))

In [47]:
tscene_outlier_treated = tscene_feat_mat
tscene_outlier_treated[channel_cols] = tscene_feat_mat[channel_cols].apply(remove_outliers_winsor, args=(0.1,0.9,))

In [48]:
outpath = os.path.join(base_dir, 'Outputs', f'outlier_{outliertreatment}', outliermeth)
basal_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{basalscene}_nonnorm.csv'), index=False)
tscene_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{targetscene}_nonnorm.csv'), index=False)
norm.to_csv(os.path.join(outpath, rf'biometric_feat_mat_scene_{targetscene}_{normtype}.csv'), index=False)